# 📝 Домашнє завдання — Модуль 3, Урок 1 (SQL)

**До уроку:** [lektsiya-1-sql-osnovy.ipynb](lektsiya-1-sql-osnovy.ipynb)

### Як виконувати
1. **Спершу запустіть комірку нижче** — вона створює базу з даними для завдань.
2. Далі під кожним завданням пишіть **SQL-запит** у виклику `show("...")` (або `conn.execute(...)`).
3. Запускайте `Shift + Enter`, дивіться результат.

## ⚙️ Підготовка — запустіть цю комірку першою

In [1]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.isolation_level = None
conn.execute("PRAGMA foreign_keys = ON")

def show(sql, params=()):
    cur = conn.execute(sql, params)
    if cur.description:
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
        w = [len(c) for c in cols]
        for r in rows:
            for i, v in enumerate(r):
                w[i] = max(w[i], len(str(v)))
        fmt = lambda vals: " | ".join(str(v).ljust(w[i]) for i, v in enumerate(vals))
        print(fmt(cols)); print("-+-".join("-"*x for x in w))
        for r in rows: print(fmt(r))

# --- Схема: магазин ---
conn.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT NOT NULL, city TEXT)")
conn.execute("CREATE TABLE products  (id INTEGER PRIMARY KEY, title TEXT NOT NULL, price REAL)")
conn.execute("""
CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    product_id  INTEGER,
    quantity    INTEGER DEFAULT 1,
    FOREIGN KEY (customer_id) REFERENCES customers(id),
    FOREIGN KEY (product_id)  REFERENCES products(id)
)
""")

conn.executemany("INSERT INTO customers VALUES (?,?,?)",
    [(1,'Ірина','Київ'),(2,'Олег','Львів'),(3,'Марія','Київ'),(4,'Богдан','Одеса')])
conn.executemany("INSERT INTO products VALUES (?,?,?)",
    [(1,'Клавіатура',800),(2,'Миша',400),(3,'Монітор',6000),(4,'Ноутбук',30000)])
conn.executemany("INSERT INTO orders (id,customer_id,product_id,quantity) VALUES (?,?,?,?)",
    [(1,1,1,2),(2,1,2,1),(3,3,3,1),(4,3,1,1),(5,2,4,1)])

print("База готова. Таблиці: customers, products, orders")
show("SELECT * FROM customers")

База готова. Таблиці: customers, products, orders
id | name   | city 
---+--------+------
1  | Ірина  | Київ 
2  | Олег   | Львів
3  | Марія  | Київ 
4  | Богдан | Одеса


### Завдання 1 — SELECT

Виведіть усі товари (`products`). _(Розділ 8.)_

In [8]:
show("SELECT title AS товари FROM products")

товари    
----------
Клавіатура
Миша      
Монітор   
Ноутбук   


### Завдання 2 — WHERE

Виведіть назви й ціни товарів, дорожчих за 500. _(Розділ 8.)_

In [9]:
show("SELECT title, price FROM products WHERE price > 1000")

title   | price  
--------+--------
Монітор | 6000.0 
Ноутбук | 30000.0


### Завдання 3 — ORDER BY

Виведіть товари, відсортовані за ціною від найдорожчого. _(Розділ 8.)_

In [12]:
show("SELECT title, price FROM products ORDER BY price DESC")

title      | price  
-----------+--------
Ноутбук    | 30000.0
Монітор    | 6000.0 
Клавіатура | 800.0  
Миша       | 400.0  


### Завдання 4 — INSERT

Додайте нового покупця (id=5, ваше ім'я, місто). Потім виведіть усіх покупців. _(Розділ 7.)_

In [ ]:
show("INSERT INTO customers (id, name, city) Values (5, 'Iryna', 'Zhovkva')")
show("SELECT * FROM customers")

id | name   | city   
---+--------+--------
1  | Ірина  | Київ   
2  | Олег   | Львів  
3  | Марія  | Київ   
4  | Богдан | Одеса  
5  | Iryna  | Zhovkva


### Завдання 5 — UPDATE

Змініть місто покупця з id=2 на `'Харків'`. Перевірте результат SELECT-ом. _(Розділ 12.)_

In [20]:
conn.execute("UPDATE customers SET city = 'Харків' WHERE id=2")
show("SELECT * FROM customers")

id | name   | city   
---+--------+--------
1  | Ірина  | Київ   
2  | Олег   | Харків 
3  | Марія  | Київ   
4  | Богдан | Одеса  
5  | Iryna  | Zhovkva


### Завдання 6 — DELETE

Видаліть покупця з id=4 (у нього немає замовлень). Виведіть покупців. _(Розділ 12.)_

In [21]:
conn.execute("DELETE FROM customers WHERE id=4")
show("SELECT name AS покупці FROM customers")

покупці
-------
Ірина  
Олег   
Марія  
Iryna  


### Завдання 7 — Агрегація

Порахуйте кількість товарів, їхню середню й максимальну ціну (одним SELECT). _(Розділ 13.)_

In [22]:
show("SELECT COUNT(*) AS кількість_товарів, AVG(price), MAX(price) FROM products")

кількість_товарів | AVG(price) | MAX(price)
------------------+------------+-----------
4                 | 9300.0     | 30000.0   


### Завдання 8 — GROUP BY

Порахуйте, скільки покупців у кожному місті. _(Розділ 13.)_

In [24]:
show("SELECT city, COUNT(*) AS покупці FROM customers GROUP BY city")

city    | покупці
--------+--------
Zhovkva | 1      
Київ    | 2      
Харків  | 1      


### Завдання 9 — JOIN

Виведіть, хто що замовив: ім'я покупця, назву товару й кількість (з'єднайте три таблиці). _(Розділ 14.)_

In [25]:
show("""SELECT c.name AS покупець, p.title AS товар, o.quantity AS кількість 
FROM orders o
JOIN customers c ON o.customer_id=c.id
JOIN products p ON o.product_id=p.id""")

покупець | товар      | кількість
---------+------------+----------
Ірина    | Клавіатура | 2        
Ірина    | Миша       | 1        
Марія    | Монітор    | 1        
Марія    | Клавіатура | 1        
Олег     | Ноутбук    | 1        


### Завдання 10 — HAVING

Покажіть міста, де більше за одного покупця. _(Розділ 13.)_

In [29]:
show("SELECT city FROM customers GROUP BY city HAVING COUNT(*)>1")
show("SELECT * FROM customers")#для перевірки

city
----
Київ
id | name  | city   
---+-------+--------
1  | Ірина | Київ   
2  | Олег  | Харків 
3  | Марія | Київ   
5  | Iryna | Zhovkva


### Завдання 11 (⭐) — VIEW

Створіть представлення `order_details` (ім'я покупця + товар + сума = price*quantity) і зробіть з нього SELECT. _(Розділи 14–15.)_

In [ ]:
# conn.execute("""CREATE VIEW order_details AS
#              SELECT c.name AS 'ім"я_покупця', p.title AS 'назва_товару', p.price*o.quantity AS 'сума'
#              FROM orders o
#              JOIN customers c ON o.customer_id=c.id
#              JOIN products p ON o.product_id=p.id
#              """)
show("SELECT * FROM order_details")

ім"я_покупця | назва_товару | сума   
-------------+--------------+--------
Ірина        | Клавіатура   | 1600.0 
Ірина        | Миша         | 400.0  
Марія        | Монітор      | 6000.0 
Марія        | Клавіатура   | 800.0  
Олег         | Ноутбук      | 30000.0


### Завдання 12 (⭐) — Захист від ін'єкції

Напишіть функцію `find_customer(name)`, що безпечно (через параметр `?`) шукає покупця за іменем. Перевірте на звичайному імені та на `"' OR '1'='1"`. _(Розділ 18.)_

In [36]:
def find_customer(name):
    return conn.execute("SELECT name FROM customers WHERE name=?", (name,)).fetchall()

print("Звичайний запит:", len(find_customer("Iryna")))
print("Спроба ін'єкції:", len(find_customer("' OR '1'='1"))) 

Звичайний запит: 1
Спроба ін'єкції: 0


---
✅ Готово? Збережіть зошит. Далі — Урок 2: бази даних із Python (SQLAlchemy, psycopg2)! 🎉